# Task 3: Random Forest tuned with Optuna
Dataset: Breast Cancer Wisconsin (Diagnostic), same `data.csv` used across this project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              ConfusionMatrixDisplay, classification_report)

## Load and clean the data

In [ ]:
df = pd.read_csv("data.csv")
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])

le = LabelEncoder()
df["diagnosis"] = le.fit_transform(df["diagnosis"])  # M -> 1, B -> 0

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Baseline: default Random Forest
Just to have a number to beat before tuning anything.

In [ ]:
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)
baseline_pred = baseline_rf.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))
print("Baseline F1:", f1_score(y_test, baseline_pred))

## Define the Optuna search space
Tuning number of trees, depth, split/leaf sizes, and how many features each split

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
    }
    clf = RandomForestClassifier(random_state=42, **params)
    score = cross_val_score(clf, X_train, y_train, cv=5, scoring="f1").mean()
    return score

## Run the study
Optimizing for F1 rather than plain accuracy since that's the metric we've been tracking across the other tasks.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="maximize", study_name="random_forest_tuning")
study.optimize(objective, n_trials=60)

print("Best params:", study.best_params)
print("Best CV F1:", study.best_value)

## Optimization history

In [ ]:
trial_numbers = [t.number for t in study.trials]
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)

plt.figure(figsize=(8, 5))
plt.scatter(trial_numbers, trial_values, alpha=0.5, label="trial score")
plt.plot(trial_numbers, best_so_far, color="crimson", label="best so far")
plt.xlabel("Trial")
plt.ylabel("CV F1 score")
plt.title("Optuna optimization history - Random Forest")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Which parameters mattered most
Optuna can rank parameter importance based on how much each one moved the objective across trials.

In [ ]:
importances = optuna.importance.get_param_importances(study)
imp_df = pd.DataFrame(importances.items(), columns=["param", "importance"]).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(imp_df["param"], imp_df["importance"], color="slateblue")
plt.xlabel("Relative importance")
plt.title("Optuna parameter importance")
plt.tight_layout()
plt.show()

## Final model with the best parameters

In [ ]:
best_rf = RandomForestClassifier(random_state=42, **study.best_params)
best_rf.fit(X_train, y_train)
best_pred = best_rf.predict(X_test)

print("Tuned accuracy:", accuracy_score(y_test, best_pred))
print("Tuned F1:", f1_score(y_test, best_pred))
print(classification_report(y_test, best_pred))

## Confusion matrix - tuned model

In [ ]:
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap="Greens")
plt.title("Tuned Random Forest - Confusion Matrix")
plt.show()

## Baseline vs tuned, side by side

In [ ]:
comparison = pd.DataFrame({
    "model": ["baseline", "optuna-tuned"],
    "accuracy": [accuracy_score(y_test, baseline_pred), accuracy_score(y_test, best_pred)],
    "f1_score": [f1_score(y_test, baseline_pred), f1_score(y_test, best_pred)],
})
comparison